In [1]:
from arango import ArangoClient, ServerVersionError, DocumentUpdateError
from datetime import datetime, timedelta
from ollama import Client
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor, as_completed
from multiprocessing import Lock
from tqdm.notebook import tqdm
from mlflow.genai import scorer
from mlflow.genai.scorers import Correctness, Guidelines

import getpass
import traceback
import json
import os
import random
import threading
import mlflow

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## Functions
##### Run Cells to Continue

In [2]:
def connect_to_arango_client(host: str):
    try:
        client = ArangoClient(hosts=host)
    except:
        print(f'{get_timestamp()} -- Could not connect to ArangoDB client at "{host}"')
        return None
    
    return client

In [3]:
def connect_to_arango_db(client: ArangoClient, db_name: str, username: str, password: str, max_retries: int = 3): 
    retries = 0
    if password is None:
        password = getpass.getpass(f'Please enter the password for user {username} for access to the {db_name} database: ')
        
    while retries < max_retries:
        try:
            db = client.db(db_name, username=username, password=password)     
            vers = db.version()
            print(f'{get_timestamp()} -- Successfully connected to database {db_name} running version {vers}.')
            return db
        except (ConnectionAbortedError, ServerVersionError) as e:
            retries += 1
            print(f'{get_timestamp()} -- Connection refused for db {db_name}.')
            password = getpass.getpass(f'Input password to try again:  ')
        except Exception as e:
            print(f'\n**** {type(e)}: {e} ****\n')
            retries += 1
            print(f'{get_timestamp()} -- Could not connect to db "{db_name}".')
    
    return None

In [4]:
def without(d, keys):
    new_d = d.copy()
    for key in keys:
        if key in d.keys():
            new_d.pop(key)
    return new_d

In [5]:
def get_timestamp(str_format: str = "%Y-%m-%d %H:%M:%S"):
    return datetime.now().strftime(str_format)

def connect_to_arango_client(host: str):
    try:
        client = ArangoClient(hosts=host)
    except:
        print(f'{get_timestamp()} -- Could not connect to ArangoDB client at "{host}"')
        return None
    
    return client


def connect_to_arango_db(client: ArangoClient, db_name: str, username: str, password: str, max_retries: int = 3): 
    retries = 0
    if password is None:
        password = getpass.getpass(f'Please enter the password for user {username} for access to the {db_name} database: ')
        
    while retries < max_retries:
        try:
            db = client.db(db_name, username=username, password=password)     
            vers = db.version()
            print(f'{get_timestamp()} -- Successfully connected to database {db_name} running version {vers}.')
            return db
        except (ConnectionAbortedError, ServerVersionError) as e:
            retries += 1
            print(f'{get_timestamp()} -- Connection refused for db {db_name}.')
            password = getpass.getpass(f'Input password to try again:  ')
        except Exception as e:
            print(f'\n**** {type(e)}: {e} ****\n')
            retries += 1
            print(f'{get_timestamp()} -- Could not connect to db "{db_name}".')
    
    return None

In [6]:
def exec_aql_query(aql, query):
    cursor = aql.execute(query)
    return cursor

In [7]:
def get_steps_for_all_ttps(aql, graph_name):
    ttp_steps_query = """
        LET step_collections = ["DevelopmentStep", "PlanningStep", "ExecutionStep"]

            FOR ttp IN TTPArtifact
                LET refs = (
                    FOR a IN INBOUND ttp GRAPH '""" + graph_name + """'    // *Artifact → TTPArtifact
                        FOR s IN INBOUND a GRAPH '""" + graph_name + """'   // *Step → *Artifact
                            LET step_type = SPLIT(s._id, "/")[0]
                            FILTER step_type IN step_collections
                            COLLECT stepType = step_type INTO grouped_artifacts = a
                            RETURN {
                                step_type: stepType,
                                reference_count: LENGTH(grouped_artifacts),
                                artifact_keys: UNIQUE(FOR g IN grouped_artifacts RETURN g._key)
                            }
                )
                RETURN {
                    ttp_key: ttp._key,
                    references_by_step_type: refs
                }
    """
    
    cursor = aql_query_graph(aql, ttp_steps_query)
    
    ttp_steps = [doc for doc in cursor]

    return ttp_steps


In [8]:
def get_sample_pairs_from_graph(db, aql, graph_name, 
                                n_samples=5, 
                                pairs_per_sample=5, 
                                src_filter_str='', 
                                pairs_filter_str='', 
                                src_collections=[], 
                                pairs_collections=[],
                                exclude_src_collections=[],
                                exclude_pairs_collections=[]
                               ):
    graph = db.graph(graph_name)

    # Get all vertex collections in the graph
    vertex_collections = graph.vertex_collections()
    if len(src_collections) == 0:
        src_collections = vertex_collections
    if len(pairs_collections) == 0:
        pairs_collections = vertex_collections
    
    src_included_collections = [coll for coll in vertex_collections if coll in src_collections and coll not in exclude_src_collections] 
    pairs_included_collections = [coll for coll in vertex_collections if coll in pairs_collections and coll not in exclude_pairs_collections]

    
    # Sample n nodes from each collection
    sampled_nodes = []
    limit_str = f'LIMIT {n_samples}' if n_samples > 0 else ''
    src_filter_str = f'FILTER {src_filter_str}' if src_filter_str != '' else ''
    pairs_filter_str = f'FILTER {pairs_filter_str}' if pairs_filter_str != '' else ''
    
    for vc in src_included_collections:
        query = f"""
        FOR doc IN {vc}
            {src_filter_str}
            SORT RAND()
            {limit_str}
            RETURN doc
        """
        result = list(aql.execute(query, batch_size=1000))
        #print(f'**** Len results: {len(result)} ****')
        sampled_nodes.extend([without(doc, ['_rev']) for doc in result])


    # Fetch all node documents (to choose random m for pairing) ---
    # Build a union query to get all documents from all vertex collections
    union_parts = [f"FOR doc IN {vc} {pairs_filter_str} RETURN doc" for vc in pairs_included_collections]
    all_docs_query = f"RETURN UNION({', '.join(union_parts)})" if len(pairs_included_collections) > 1 else union_parts[0]
    #print(f'all_docs_query:\n{all_docs_query}')

    cursor = list(aql.execute(all_docs_query, batch_size=10000))
    #print(isinstance(list(cursor)[0], dict))
    all_docs = [without(doc, ['_rev']) for batch in cursor for doc in batch] if len(pairs_included_collections) > 1 else [without(doc, ['_rev']) for doc in cursor]
    #print(all_docs[0])
    all_docs_by_id = {doc["_id"]: doc for doc in all_docs}
    
    # Build pairings
    pairings = []
    
    for node in sampled_nodes:
        node_id = node["_id"]
        
        # Exclude the current node from the pool
        candidates = [doc for doc_id, doc in all_docs_by_id.items() if doc_id != node_id]
        
        # Randomly sample m candidates
        if len(candidates) < pairs_per_sample or pairs_per_sample < 1:
            sampled = candidates
        else:
            sampled = random.sample(candidates, pairs_per_sample)

        #print(f'*** len(pairs): {len(sampled)} ***')
        pairings = pairings + [{'src_node': node, 'pair_node': sample} for sample in sampled]

    return pairings


In [9]:
def get_graph_edges(db, aql, graph_name, include_node_docs=False):
    graph = db.graph(graph_name)
    valid_edges = []
    # Loop through edge definitions in the graph
    for ed in graph.edge_definitions():
        edge_collection = ed["edge_collection"]
        #print(ed)
        from_colls = ed["from_vertex_collections"]
        to_colls = ed["to_vertex_collections"]
    
        # Build AQL query for this edge definition
        # Accept any valid (from, to) pair from the defined collections
        filters = []
        for from_coll in from_colls:
            for to_coll in to_colls:
                filters.append(
                    f"(IS_SAME_COLLECTION(edge._from, '{from_coll}') && IS_SAME_COLLECTION(edge._to, '{to_coll}'))"
                )
        
        filter_clause = " || ".join(filters)
    
        aql_query = f"""
        FOR edge IN {edge_collection}
            FILTER {filter_clause}
            RETURN edge
        """
    
        result = list(aql.execute(aql_query, batch_size=1000))
        if include_node_docs:
            for edge in result:
                #print(edge)
                edge['_from_node'] = db.collection(edge['_from'].split('/')[0]).get(edge['_from'])
                edge['_to_node'] = db.collection(edge['_to'].split('/')[0]).get(edge['_to'])
                edge['type'] = edge_collection
            result = [edge for edge in result if edge['_from_node'] is not None and edge['_to_node'] is not None]
            
        valid_edges.extend(result)
    return valid_edges

In [10]:
def get_graph_nodes(db, aql, graph_name, include_node_docs=False):
    graph = db.graph(graph_name)
    valid_nodes = []
    # Loop through edge definitions in the graph
    for vertex_collection in graph.vertex_collections():
    
        # Build AQL query for this edge definition
        # Accept any valid (from, to) pair from the defined collections
    
        aql_query = f"""
        FOR node IN {vertex_collection}
            RETURN node
        """
    
        result = list(aql.execute(aql_query, batch_size=1000))
        
        valid_nodes.extend(result)
    return valid_nodes

In [11]:
def update_edgecoll_fields(db, aql, rename_fields={}, collections=[]):
    # --- Get all edge collections in the DB ---
    all_collections = db.collections()
    #rint(all_collections)
    edge_collections = [c["name"] for c in all_collections if c["type"] == 'edge'] if len(collections) == 0 else collections
    print(f"Found {len(edge_collections)} edge collections in the database.")
    
    # Build the merge dict part: { new_field: doc.old_field, ... }
    merge_fields = ", ".join(
        f"{new_field}: doc.{old_field}" for old_field, new_field in rename_fields.items()
    )

    # Build list of old fields to unset
    old_fields_list = ", ".join(f"'{old_field}'" for old_field in rename_fields.keys())
    
        # --- Run update on each edge collection ---
    for ec in edge_collections:
        print(f"Updating edge collection: {ec}")
        aql_query = f"""
        FOR doc IN {ec}
            FILTER {" || ".join(f"HAS(doc, '{old_field}')" for old_field in rename_fields.keys())}
            LET updated = UNSET(MERGE(doc, {{
                {merge_fields}
            }}), [{old_fields_list}])
            UPDATE doc WITH updated IN {ec}
        """
        print(aql_query)
        aql.execute(aql_query)
        print(f"Done updating {ec}")
    
    print("All edge documents updated.")

In [12]:
def show_n_ttps(ttps: list, n: int = 5, sort_on: str = None, from_end: str = 'top'):
    if from_end == 'top':
        if sort_on is not None:
            ttps = sorted(ttps, key=lambda x: x[sort_on], reverse=True)
        ttp_slice = ttps[:n]
        header_str_end = 'Top'
    elif from_end == 'bot':
        if sort_on is not None:
            ttps = sorted(ttps, key=lambda x: x[sort_on], reverse=False)
        ttp_slice = ttps[-n:]
        header_str_end = 'Bottom'
    else:
        print(f'Invalid value for from_end: {from_end}. Expected "top" or "bot".')

    header_str = f'********** {header_str_end} {n} TTPs **********'
    h_str_len = len(header_str)
    print(header_str)
    print('-'*h_str_len)
    print(f'   {"TTP":<13}|  {"References":<10}')
    print('-'*h_str_len)
    for ttp in ttp_slice:
        print(f'   {ttp["ttp_key"]:<13}|  {ttp["refs"]:>9}')

In [13]:
def plot_ttp_artifact_histogram(data: list[dict], save_to_file: bool = False, save_file: str = 'ttp_hist.png'):
    # Extract all unique step_types across all entries
    step_types = sorted({
        step['step_type']
        for entry in data
        for step in entry.get('references_by_step_type', [])
    })

    ttp_keys = [entry['ttp_key'] for entry in data]
    n = len(ttp_keys)
    x = np.arange(n)  # x positions for each ttp_key

    width = 0.5 / max(len(step_types), 1)  # dynamic bar width based on number of step_types

    # Prepare counts per step_type aligned with ttp_keys
    counts_per_step = {step: [] for step in step_types}
    total_counts = []
    for entry in data:
        step_count_map = {step['step_type']: step['reference_count'] for step in entry.get('references_by_step_type', [])}
        total = 0
        for step in step_types:
            count = step_count_map.get(step, 0)
            counts_per_step[step].append(step_count_map.get(step, 0))
            total += count
        total_counts.append(total)

    fig, ax = plt.subplots(figsize=(max(8, n), 5))

    # Plot bars for each step_type with proper offsets
    for i, step in enumerate(step_types):
        offset = (i - (len(step_types) - 1) / 2) * width
        ax.bar(x + offset, counts_per_step[step], width, label=step)

    # total counts bar
    ax.plot(x, total_counts, color='orchid', label='Total References', linewidth=0.7, linestyle='--')

    
    # Formatting
    ax.set_xlabel('TTP')
    ax.set_ylabel('Reference Count')
    ax.set_title('Reference Counts per Step Type for each TTP and Total per TTP')
    ax.set_xticks(x)
    ax.set_xticklabels(ttp_keys, rotation=45, ha='right')
    ax.legend(title="Step Type")
    ax.grid(axis='y', linestyle='--', alpha=0.3)
    plt.tight_layout()
    if save_to_file:
        fig.savefig(save_file)
    plt.show()

    


In [14]:
def almost_in_list(truth_list, edge, possible_maps={'null_key': {'suggested_vals': [], 'truth_vals': []}}):
    if edge in truth_list:
        return True
    for key in possible_maps.keys():
        for sval in possible_maps[key]['suggested_vals']:
            for tval in possible_maps[key]['truth_vals']:
                edge_copy = edge.copy()
                edge_copy[key] = tval if edge[key] == sval else edge[key]
                if edge_copy in truth_list:
                    return True
    return False

In [15]:
def get_wrong_keyvals(truth_list, edge, possible_maps={'null_key': {'suggested_vals': [], 'truth_vals': []}}):
    wrong_kvs = {}
    found_match = False
    if edge is None or len(edge.keys()) == 0:
        return {}
    else:
        for t_edge in truth_list:
            if t_edge['_from'] == edge['_from'] and t_edge['_to'] == edge['_to']:
                found_match = True
                for key in t_edge.keys():
                    if key not in edge:
                        wrong_kvs[key] = {'suggested_val': None, 'truth_val': t_edge[key]}
                    elif edge[key] != t_edge[key] and not (key in possible_maps and edge[key] in possible_maps[key]['suggested_vals'] and t_edge[key] in possible_maps[key]['truth_vals']):
                        wrong_kvs[key] = {'suggested_val': edge[key], 'truth_val': t_edge[key]}
    if not found_match:
        wrong_kvs = {
            key: {
                'suggested': edge[key],
                'truth_val': None
            } for key in edge.keys()
        }
    return wrong_kvs

In [16]:
def prompt_and_response(client, model, prompt, sys_set=None, quiet=False, temperature=0.8, options={}, stream=True):
    p_start = datetime.now()
    if 'temperature' not in options:
        options['temperature'] = temperature
    response = ''
    try:
        for part in client.generate(model=model, prompt=prompt, stream=stream, options=options):
            response += part['response']
            if not quiet:
                print(part['response'], end='', flush=True)
    except Exception as e:
        print(f'ERROR: {e}\nSkipping prompt...')
        traceback.format_exc(e)
        return None
    p_end = datetime.now() - p_start
    if not quiet:
        print(f'Prompt took {p_end}')

    return response

In [17]:
def part1_prompt(edge_pairs, oll_client, model='gemma3:27b-it-qat', quiet=True, options={'temperature': 0.8}, src_node_key='src_node', pair_node_key='pair_node'):
    part1_responses = []
    max_pairs_listed = 10
    prompts_start = datetime.now()
    pairings = [
        {
            'src_node': edge[src_node_key], 
            'pair_node': edge[pair_node_key]
        } for edge in edge_pairs if edge[src_node_key] is not None and edge[pair_node_key] is not None
    ]
    
    for node_pair in tqdm(pairings, desc='Prompt 1', leave=False):
        if not quiet:
            print(json.dumps(node_pair, indent=4))
        #print(node_pair["src_node"]["_id"])
        
        prompt = f"""Act as a data analyzer. Consider the following data for a graph network, as a source node and a destination node:
        {json.dumps(node_pair)}
        
        After considering attributes, values, and contexts, answer the following question: 
        Should an edge exist between these two nodes? Err on the side of skepticism.
        Answers should be in the form of True or False.
        
        Do not provide any conversation, only the boolean True/False as your answer.
        """
        #print()
        #print(len(prompt))
        
        response = prompt_and_response(oll_client, model, prompt, quiet=quiet, options=options)
        if response is not None:
            #print(type(response))
            j_response = {
                'src_node': node_pair['src_node'],
                'pair_node': node_pair['pair_node'],
                'is_connected': response
            }
            part1_responses.append(j_response)
        else:
            print(f'Response for {node_pair['src_node']['_id']} -> {node_pair['pair_node']['_id']} is None.')
        
    
    prompts_total = datetime.now() - prompts_start
    if not quiet:
        print(f'***** Prompts took a total of: {prompts_total} *****')

    return part1_responses

In [18]:
def part2_prompt(part1_responses, oll_client, model='llama3.3:70b', quiet=True, options={'temperature': 0.8}):
    part2_responses = []
    max_pairs_listed = 10
    prompts_start = datetime.now()
    pairings = [
        {
            'src_node': edge['src_node'], 
            'pair_node': edge['pair_node']
        } for edge in part1_responses if edge['is_connected'] #get_graph_edges(db, aql, graph_name, include_node_docs=True) if edge['_from_node'] is not None and edge['_to_node'] is not None
    ]
    
    for node_pair in tqdm(pairings, desc='Prompt 2', leave=False):
        if not quiet:
            print(json.dumps(node_pair, indent=4))
        #print(node_pair["src_node"]["_id"])
        
        prompt = f"""
        Act as a data analyzer for cybersecurity graph relationships.
 
        Given these two nodes:
        {json.dumps(node_pair, indent=2)}
         
        Analyze their attributes to determine the relationship type.
         
        **Edge Type Definitions:**
        - LEADS_TO: Sequential relationship where source directly precedes destination in time/workflow
        - REFERENCES: Source cites or refers to destination for context (not execution)
        - PRODUCES: Source creates destination as an output artifact
        - CONTAINS: Destination is a part of source
        - COLLABORATION_WITH: Destination collaborated with another team to produce Source artifact
         
        **Analysis Instructions:**
        1. Identify matching field values (e.g., ttp_ids == tid)
        2. Consider semantic relationships based on node types and related field values
        3. Choose the most specific edge type that applies. Choose from the Edge Type Definitions above.
        4. If no clear connection exists, return {{"explanation": "NO CONNECTION", "type": "None", "src_attr": "None", "dest_attr": "None"}}
         
        Return a single JSON object:
        {{"src_attr": "<source_field>", "dest_attr": "<dest_field>", "type": "<EDGE_TYPE>", "explanation": "<why these specific fields connect>"}}
         
        Example for matching IDs:
        {{"src_attr": "ttp_ids", "dest_attr": "tid", "type": "REFERENCES", "explanation": "Log entry technique_id T1059 matches TTP tid, indicating the log implements this technique"}}
         
        JSON response only, no additional text:
        """
        #print()
        #print(len(prompt))
        
        response = prompt_and_response(oll_client, model, prompt, quiet=quiet, options=options)
        if response is not None:
            #print(response)
            try:
                if isinstance(response, list):
                    for r in response:
                        r = json.loads(r.replace('```json', '').replace('```', '').replace('</end_of_turn>', '')) 
                        r['_from'] = node_pair['src_node']['_id']
                        r['_to'] = node_pair['pair_node']['_id']
                        r['src_node'] = node_pair['src_node']
                        r['pair_node'] = node_pair['pair_node']
                        part2_responses.append(r)
                else:
                    response = json.loads(response.replace('```json', '').replace('```', '').replace('</end_of_turn>', '')) 
                    response['_from'] = node_pair['src_node']['_id']
                    response['_to'] = node_pair['pair_node']['_id']
                    response['src_node'] = node_pair['src_node']
                    response['pair_node'] = node_pair['pair_node']
                    part2_responses.append(response)
            except:
                print(f'ERROR: unknown response format: {response}')
        else:
            print(f'Response for {node_pair['src_node']['_id']} -> {node_pair['pair_node']['_id']} is None.')
        
    
    prompts_total = datetime.now() - prompts_start
    if not quiet:
        print(f'***** Prompts took a total of: {prompts_total} *****')

    return part2_responses

In [19]:
def part3_prompt(part2_responses, oll_client, model='gemma3:27b-it-qat', quiet=True, options={'temperature': 0.8}):
    part3_responses = []
    max_pairs_listed = 10
    prompts_start = datetime.now()
    pairings = [
        {
            'src_node': edge['src_node'], 
            'pair_node': edge['pair_node']
        } for edge in part2_responses #get_graph_edges(db, aql, graph_name, include_node_docs=True) if edge['_from_node'] is not None and edge['_to_node'] is not None
    ]
    
    for i in tqdm(range(0, len(pairings)), desc='Prompt 3', leave=False):
        node_pair = pairings[i]
        if not quiet:
            print(f'Part2 response: {json.dumps(part2_responses[i], indent=4)}\n')
            print(json.dumps(node_pair, indent=4))
        #print(node_pair["src_node"]["_id"])
        
        prompt = f"""
        Act as a data analyzer. Consider the following data for a graph network, as a source node and a destination node:
        {json.dumps(node_pair)}
        
        Rate the strength and accuracy of a potential connection from 1-10, with 1 meaning no connection should have been made or the connection is incorrect or invalid and 10 being a very strong connection.
        Answers should be in the form of an integer between 1 and 10.
        
        Example output: "9"
        Think through this problem and then provide me ONLY the conclusion. Do not show your reasoning process in the final answer.    
        """
        #print()
        #print(len(prompt))
        
        response = prompt_and_response(oll_client, model, prompt, quiet=quiet, options=options)
        if response is not None:
            #print(response)
            j_response = {
                'src_node': node_pair['src_node'],
                'pair_node': node_pair['pair_node'],
                '_key': node_pair['src_node']['_id'].replace('/', ':') + '_to_' + node_pair['pair_node']['_id'].replace('/', ':'),
                '_from': node_pair['src_node']['_id'],
                '_to': node_pair['pair_node']['_id'],
                'type': part2_responses[i]['type'],
                'explanation': part2_responses[i]['explanation'],
                'src_attr': part2_responses[i]['src_attr'],
                'dest_attr': part2_responses[i]['dest_attr'],
                'conn_strength': int(response)
            }
            part3_responses.append(j_response)
        else:
            print(f'Response for {node_pair['src_node']['_id']} -> {node_pair['pair_node']['_id']} is None.')

        
    
    prompts_total = datetime.now() - prompts_start
    if not quiet:
        print(f'***** Prompts took a total of: {prompts_total} *****')

    return part3_responses

In [20]:
def append_json_to_file(new_entry, file_path, write_lock):
    with write_lock:
        # Check if file exists
        with open(file_path, 'a') as file:
            file.write(json.dumps(new_entry, default=str) + '\n')

In [21]:
def genai_composite_score(total_expected, avg_exact_match, avg_partial_match, avg_avg_inc_kv_counts, exact_match_range, partial_match_range, w_e=1.0, w_p=0.5, w_i=0.4, alpha=0.7):
    # --- Accuracy ---
    accuracy = (
        w_e * avg_exact_match +
        w_p * avg_partial_match -
        w_i * (avg_avg_inc_kv_counts / total_expected)
    )
    accuracy = max(0, min(1, accuracy)) # clip to [0, 1]
    
    # --- Consistency ---
    consistency = 1 - 0.05 * (exact_match_range + partial_match_range)
    consistency = max(0, min(1, consistency)) # clip to [0, 1]
    
    # --- Composite ---
    score = alpha * accuracy + (1 - alpha) * consistency
    
    return {
        "weighted_accuracy": accuracy,
        "consistency": consistency,
        "genai_score": score
    }


In [22]:
def get_edge_if_exists(edge_coll, key):
    if edge_coll.has(key):
        return edge_coll.get(key)
    return None

In [23]:
def verify_edges(whole_suggested_edges, db, auto_accept_partial=False):
    verified_edges = []
    denied_edges = []
    y_responses = ['y', 'yes']
    for edge_set in tqdm(whole_suggested_edges):
        print(f'{json.dumps(edge_set, indent=4, default=str)}')
        sugg_edge = edge_set
        edge_coll = db.collection(sugg_edge['type'])
        existing_edge = get_edge_if_exists(edge_coll, sugg_edge['_key'])
        if existing_edge is not None:
            if existing_edge['src_attr'] == sugg_edge['src_attr'] and existing_edge['dest_attr'] == sugg_edge['dest_attr']:
                print('Edge already exists as-is.')
            else:
                print(f'Edge already exists, but one of "src_attr" or "dest_attr" are different ({existing_edge["src_attr"]}, {existing_edge["dest_attr"]}).')
            if auto_accept_partial:
                verified_edges.append(sugg_edge)
                print('\n\n')
                continue
        
        verification = input('Verify this suggested edge? (y/n)  ')
        print('\n\n')
        if verification.lower() in y_responses:
            verified_edges.append(sugg_edge)
        else:
            denied_edges.append(sugg_edge)
        
    return verified_edges, denied_edges

In [71]:
def add_verified_edges(verified_edges, db):
    for edge in tqdm(verified_edges):
        try:
            collection = db.collection(edge['type'])
        except:
            print('Skipping...')
            continue
        try:
            collection.update(without(edge, ['src_node', 'pair_node', 'status']))
        except DocumentUpdateError:
            collection.insert(without(edge, ['src_node', 'pair_node', 'status']))

In [25]:
def suggest_edges(edge_pairs, oll_client, experiment_params, model, model2, temp, temp2, write_lock, src_node_key='_from_node', pair_node_key='_to_node'):    

    run_params = {
        'model1': model,
        'model2': model2,
        'temp1': temp,
        'temp2': temp2,
        'partial_correct_threshold': experiment_params['partial_correct_threshold'],
        'quiet': experiment_params['quiet'],
        'json_output_file': experiment_params['json_output_file']
    }
    
    
    experiment_results = {
            'model1': model,
            'model2': model2,
            'temp1': temp,
            'temp2': temp2,
        }
    partial_correct_threshold = experiment_params['partial_correct_threshold']
    quiet = experiment_params['quiet']
    

    if not quiet:
        print(f'Running combination: ({model}, {model2}, {temp}, {temp2})')
    
    mlflow.set_experiment('Model Runs')
    with mlflow.start_run(run_name=f'{model}-{temp}_{model2}-{temp2}__{datetime.now()}'): 
        mlflow.log_params(run_params)
                
        start_time = datetime.now()
        part1_responses = part1_prompt(edge_pairs, oll_client, model=model, quiet=quiet, options={'temperature': temp}, src_node_key=src_node_key, pair_node_key=pair_node_key)
        part2_responses = part2_prompt(part1_responses, oll_client, model=model2, quiet=quiet, options={'temperature': temp2})
        #print(part2_responses[0])
        part3_responses = part3_prompt(part2_responses, oll_client, model=model, quiet=quiet, options={'temperature': temp})
        
        suggested_edges = sorted([without(edge, ['_id', 'status']) for edge in part3_responses], key=lambda x: (x['_from'], x['_to']))
        
        #print(suggested_edges[0])
        
        
        end_time = datetime.now() - start_time

        iter_results = {
                'total_suggested_edges': len(suggested_edges),
                'total_runtime': end_time,
                'part1_responses': part1_responses,
                'part2_responses': part2_responses,
                'part3_responses': part3_responses
            }
            
        iter_results_mlf = iter_results.copy()
        iter_results_mlf['total_runtime'] = float(iter_results_mlf['total_runtime'].total_seconds())
        experiment_results.update(iter_results_mlf)                    
                
        #print(experiment_results)
        append_json_to_file(experiment_results, experiment_params['json_output_file'], write_lock)
        
        mlflow.log_metrics(without(experiment_results, ['iterations', 'model1', 'model2', 'temp1', 'temp2', 'part1_responses', 'part2_responses', 'part3_responses']))

    return experiment_results

In [26]:
def run_grid(edge_pairs, truth_edge_pairs, oll_client, experiment_params, model, model2, temp, temp2, write_lock, pbar, existing_data={}):     
    run_params = {
        'model1': model,
        'model2': model2,
        'temp1': temp,
        'temp2': temp2,
        'iterations': experiment_params['iterations'],
        'partial_correct_threshold': experiment_params['partial_correct_threshold'],
        'quiet': experiment_params['quiet'],
        'json_output_file': experiment_params['json_output_file']
    }  
    
    experiment_results = {
            'model1': model,
            'model2': model2,
            'temp1': temp,
            'temp2': temp2,
            'iterations': []
        }
    iterations = experiment_params['iterations']
    partial_correct_threshold = experiment_params['partial_correct_threshold']
    quiet = experiment_params['quiet']
    
    if len(existing_data) > 0:
        matches = [d for d in existing_data if all(d.get(k) == v for k,v in without(experiment_results, ['iterations']).items())]
            
        if len(matches) > 0:
            #print('test')
            if not quiet:
                print(f'Combination already exists in experiments: ({model}, {model2}, {temp}, {temp2})')
            return None

    if not quiet:
        print(f'Running combination: ({model}, {model2}, {temp}, {temp2})')

    mlflow.set_experiment('Model Grid - 102825')
    with mlflow.start_run(run_name=f'{model}-{temp}_{model2}-{temp2}__{datetime.now()}') as parent_run:
        mlflow.log_params(run_params)
        
        for i in tqdm(range(0, iterations), desc='Iterations', leave=False):
            with mlflow.start_run(run_name=f'Iter-{i}_{model}-{temp}_{model2}-{temp2}__{datetime.now()}', nested=True, parent_run_id=parent_run.info.run_id) as child_run:
                mlflow.log_params(run_params)
                
                start_time = datetime.now()
                part1_responses = part1_prompt(edge_pairs, oll_client, model=model, quiet=quiet, options={'temperature': temp}, src_node_key='_from_node', pair_node_key='_to_node')
                part2_responses = part2_prompt(part1_responses, oll_client, model=model2, quiet=quiet, options={'temperature': temp2})
                #print(part2_responses[0])
                part3_responses = part3_prompt(part2_responses, oll_client, model=model, quiet=quiet, options={'temperature': temp})
                
                truth_edges = sorted([without(edge, ['_rev', 'source_attr', 'source_field', 'destination_field', '_key', '_id', '_from_node', '_to_node']) for edge in truth_edge_pairs if edge['_from_node'] is not None and edge['_to_node'] is not None], key=lambda x: (x['_from'], x['_to']))
        
                #print(truth_edges[0])
                
                #print(responses[0])
                #print(json.loads(responses.replace('```json', '').replace('```', '')))
                    
                suggested_edges = sorted([without(edge, ['explanation', '_key', '_id', 'conn_strength', 'src_node', 'pair_node', 'status']) for edge in part3_responses if edge['explanation'].upper() != "NO CONNECTION"], key=lambda x: (x['_from'], x['_to']))
        
                #print(suggested_edges[0])
                
                truth_list = [almost_in_list(truth_edges, edge, possible_maps={'dest_attr': {'suggested_vals': ['_key', '_id', 'id', 'name', 'tid'], 'truth_vals': ['_key', 'id', '_id', 'name', 'tid']}}) for edge in suggested_edges ]
        
                wrong_kvs = [get_wrong_keyvals(truth_edges, edge, possible_maps={'dest_attr': {'suggested_vals': ['_key', '_id', 'id', 'name', 'tid'], 'truth_vals': ['_key', 'id', '_id', 'name', 'tid']}}) for edge in suggested_edges ]
                
                incorrect_keyval_counts = [len(kvs.keys()) for kvs in wrong_kvs]
        
                whole_pairs = [
                    {
                        'src_node': pair['src_node'],
                        'pair_node': pair['pair_node'],
                        'suggested_edge': next((s for s in suggested_edges if s['_from'] == pair['src_node']['_id'] and s['_to'] == pair['pair_node']['_id']), None),
                        'truth_edge': next((t for t in truth_edges if t['_from'] == pair['src_node']['_id'] and t['_to'] == pair['pair_node']['_id']), None)
                    } for pair in part3_responses
                ]
                whole_pairs = [
                    {
                        'src_node': pair['src_node'],
                        'pair_node': pair['pair_node'],
                        'suggested_edge': pair['suggested_edge'],
                        'truth_edge': pair['truth_edge'],
                        'wrong_kvs': get_wrong_keyvals([pair['truth_edge']], pair['suggested_edge'], possible_maps={'dest_attr': {'suggested_vals': ['_key', '_id', 'id', 'name', 'tid'], 'truth_vals': ['_key', 'id', '_id', 'name', 'tid']}})
                    } for pair in whole_pairs
                ]
                exact_match_pairs = [pair for pair in whole_pairs if almost_in_list([pair['truth_edge']], pair['suggested_edge'], possible_maps={'dest_attr': {'suggested_vals': ['_key', '_id', 'id', 'name', 'tid'], 'truth_vals': ['_key', 'id', '_id', 'name', 'tid']}})]
                partial_correct_pairs = [pair for pair in whole_pairs if len(pair['wrong_kvs'].keys()) <= partial_correct_threshold and len(pair['wrong_kvs'].keys()) > 0]
                incorrect_pairs = [pair for pair in whole_pairs if pair not in exact_match_pairs and pair not in partial_correct_pairs]
                
                if not quiet:
                    print(f'# Total Expected: {len(truth_edges)}\n\n# Total: {len(truth_list)}\n  # Correct: {truth_list.count(True)}\n  # Incorrect: {truth_list.count(False)}')
                
                end_time = datetime.now() - start_time
        
                iter_results = {
                        'total_expected': len(truth_edge_pairs),
                        'total_exact_match': len(exact_match_pairs),
                        'incorrect_keyval_counts': incorrect_keyval_counts,
                        'inc_kvs': wrong_kvs,
                        'avg_inc_kv_counts': sum(incorrect_keyval_counts) / len(incorrect_keyval_counts),
                        'max_inc_kv_counts': max(incorrect_keyval_counts),
                        'min_inc_kv_counts': min(incorrect_keyval_counts),
                        'total_incorrect': len(incorrect_pairs),
                        'total_missed': len(truth_edges) - (len(incorrect_pairs) + len(exact_match_pairs) + len(partial_correct_pairs)),
                        'total_partial_correct': len(partial_correct_pairs),
                        'total_expected': len(truth_edges),
                        'total_runtime': end_time,
                        'exact_match_pairs': exact_match_pairs,
                        'incorrect_pairs': incorrect_pairs,
                        'partial_correct_pairs': partial_correct_pairs,
                        'part1_responses': part1_responses,
                        'part2_responses': part2_responses,
                        'part3_responses': part3_responses
                    }

                iter_results_mlf = iter_results.copy()
                iter_results_mlf['total_runtime'] = float(iter_results_mlf['total_runtime'].total_seconds())
                experiment_results['iterations'].append(iter_results)
                mlflow.log_metrics(without(iter_results_mlf, ['exact_match_pairs', 'incorrect_pairs', 'partial_correct_pairs', 'inc_kvs', 'incorrect_keyval_counts', 'part1_responses', 'part2_responses', 'part3_responses']))
                
                if i == iterations - 1:
                    total_corrects = [it['total_exact_match'] for it in experiment_results['iterations']]
                    total_partials = [it['total_partial_correct'] for it in experiment_results['iterations']]
                    total_incorrects = [it['total_incorrect'] for it in experiment_results['iterations']]
                    total_runtimes = [it['total_runtime'] for it in experiment_results['iterations']]
                    max_inc_kv_cts = [it['max_inc_kv_counts'] for it in experiment_results['iterations']]
                    min_inc_kv_cts = [it['min_inc_kv_counts'] for it in experiment_results['iterations']]
                    total_misseds = [it['total_missed'] for it in experiment_results['iterations']]         
                    avg_inc_kv_counts_list = [it['avg_inc_kv_counts'] for it in experiment_results['iterations']]
                    
                    experiment_results['total_expected'] = len(truth_edge_pairs)
                    experiment_results['avg_exact_match'] = sum(total_corrects) / (i+1)
                    experiment_results['max_exact_match'] = max(total_corrects)
                    experiment_results['min_exact_match'] = min(total_corrects)
                    experiment_results['exact_match_range'] = max(total_corrects) - min(total_corrects)
                    experiment_results['avg_partial_correct'] = sum(total_partials) / (i+1)
                    experiment_results['max_partial_correct'] = max(total_partials)
                    experiment_results['min_partial_correct'] = min(total_partials)
                    experiment_results['partial_correct_range'] = max(total_partials) - min(total_partials)
                    experiment_results['avg_incorrect'] = sum(total_incorrects) / (i+1)
                    experiment_results['max_incorrect'] = max(total_incorrects)
                    experiment_results['min_incorrect'] = min(total_incorrects)
                    experiment_results['incorrect_range'] = max(total_incorrects) - min(total_incorrects)
                    experiment_results['avg_missed'] = sum(total_misseds) / (i+1)
                    experiment_results['max_missed'] = max(total_misseds)
                    experiment_results['min_missed'] = min(total_misseds)
                    experiment_results['missed_range'] = max(total_misseds) - min(total_misseds)
                    experiment_results['avg_avg_inc_kv_counts'] = sum(avg_inc_kv_counts_list) / len(avg_inc_kv_counts_list)
                    experiment_results['max_inc_kv_counts'] = max(max_inc_kv_cts)
                    experiment_results['min_inc_kv_counts'] = min(min_inc_kv_cts)
                    experiment_results['inc_kv_counts_range'] = max(max_inc_kv_cts) - min(min_inc_kv_cts)
                    experiment_results['avg_runtime'] = float((sum(total_runtimes, timedelta(0)) / (i+1)).total_seconds())
                    experiment_results['max_runtime'] = float((max(total_runtimes)).total_seconds())
                    experiment_results['min_runtime'] = float((min(total_runtimes)).total_seconds())
                    
                
        #print(experiment_results)
        append_json_to_file(experiment_results, experiment_params['json_output_file'], write_lock)
        metrics = genai_composite_score(
                experiment_results['total_expected'],
                experiment_results['avg_exact_match'],
                experiment_results['avg_partial_correct'],
                experiment_results['avg_avg_inc_kv_counts'],
                experiment_results['exact_match_range'],
                experiment_results['partial_correct_range'],
                w_e=1.0,
                w_p=0.5,
                w_i=0.2,
                alpha=0.7
            )
        experiment_results.update(metrics)
        mlflow.log_metrics(without(experiment_results, ['iterations', 'model1', 'model2', 'temp1', 'temp2']))
    
    return experiment_results

## Variables and Setup

In [65]:
############ USER VARIABLES ############
ar_host = 'http://localhost:8529'
db_name = 'DB_318'
username = 'root'
graph_name = '318_Processes'

oll_host = 'http://10.10.80.99:4001'
GEMMA3_27B_MODEL = 'gemma3:27b-it-qat'
GEMMA3_12B_MODEL = 'gemma3:12b-it-qat'
LLAMA3_3_70B_MODEL = 'llama3.3:70b'
MAGISTRAL_24B_MODEL = 'magistral:24b'
CODELLAMA_70B_MODEL = 'codellama:70b'
DEEPSEEK_R1_70B_MODEL = 'deepseek-r1:70b'
GPT_OSS_120B_MODEL = 'gpt-oss:120b'

n_samples = 5
pairs_per_sample = 20
src_filter_str = ''
pairs_filter_str = ''
src_collections = []
pairs_collections = []
exclude_src_collections = ['Process', 'PlanningStep', 'DevelopmentStep', 'ExecutionStep', 'TTPArtifact']
exclude_pairs_collections = ['Process', 'PlanningStep', 'DevelopmentStep', 'ExecutionStep']
conn_str_thresh = 7

mlflow_scorers = [Correctness, Guidelines(name='is_english', guidelines='The answer must be in English.')]
########## END USER VARIABLES ##########

In [28]:
password = getpass.getpass(f'Please enter the password for user {username} for access to the {db_name} database: ')

Please enter the password for user root for access to the DB_318 database:  ········


In [29]:
ar_client = connect_to_arango_client(ar_host)
db = connect_to_arango_db(ar_client, db_name, username, password)
aql = db.aql
oll_client = Client(host=oll_host)

2025-11-17 12:47:35 -- Successfully connected to database DB_318 running version 3.12.5-2.


In [28]:
export_nodes = get_graph_nodes(db, aql, graph_name)
export_edges = get_graph_edges(db, aql, graph_name)
print(len(export_nodes), len(export_edges))
export_data = {'nodes': export_nodes, 'edges': export_edges}

786 120


In [29]:
with open('graph_export_102825.json', 'w') as file:
    json.dump(export_data, file, indent=4)

In [54]:
pairings = get_sample_pairs_from_graph(db, aql, graph_name, 
                                       n_samples=n_samples, 
                                       pairs_per_sample=pairs_per_sample,
                                       src_filter_str=src_filter_str,
                                       pairs_filter_str=pairs_filter_str,
                                       src_collections=src_collections,
                                       pairs_collections=pairs_collections,
                                       exclude_src_collections=exclude_src_collections,
                                       exclude_pairs_collections=exclude_pairs_collections
                                      )

pairing_ids = [{'src_id': pair['src_node']['_id'], 'pair_id': pair['pair_node']['_id']} for pair in pairings]
total_pairs = len(pairing_ids)
print(f'Total combinations: {total_pairs}')
#print(json.dumps(pairing_ids, indent=4))

Total combinations: 800


## Run Experiment Table

In [35]:
#mlflow.openai.autolog()
mlflow.set_tracking_uri('http://localhost:5000')

edge_pairs = get_graph_edges(db, aql, graph_name, include_node_docs=True)
#edge_pairs = [edge for edge in edge_pairs if edge['_id'] == 'REFERENCES/20005']
truth_edge_pairs = edge_pairs # get_graph_edges(db, aql, graph_name, include_node_docs=True)


experiment_params = {
    'model1_models': [GPT_OSS_120B_MODEL, GEMMA3_27B_MODEL, LLAMA3_3_70B_MODEL],
    'model2_models': [GPT_OSS_120B_MODEL, GEMMA3_27B_MODEL, LLAMA3_3_70B_MODEL],
    'temp1_temperatures': [0.2, 0.5, 0.8, 1],
    'temp2_temperatures': [0.2, 0.5, 0.8, 1],
    'iterations': 5,
    'partial_correct_threshold': 1, # The number of fields that can be wrong to be considered partially correct
    'quiet': True,
    'json_output_file': './grid_results/grid_results_102925_01.json'
}
total_combos = len(experiment_params["model1_models"]) * len(experiment_params["temp1_temperatures"]) * len(experiment_params["model2_models"]) * len(experiment_params["temp2_temperatures"])
print(f'Total Combinations: {total_combos}, Total Iterations: {experiment_params["iterations"] * total_combos}')

try:
    existing_data = pd.read_json(experiment_params['json_output_file'], lines=True)
    existing_data = existing_data[['model1', 'model2', 'temp1', 'temp2']]
    existing_data = existing_data.to_dict(orient='records')
except:
    existing_data = {}

#print(existing_data)
experiments = []
futures = []

MAX_THREADS = 1
write_lock = threading.Lock()

if MAX_THREADS == 1:
    with tqdm(total=total_combos, desc='Grid Combos') as pbar:
        for model in experiment_params['model1_models']:
            for temp in experiment_params['temp1_temperatures']:
                for model2 in experiment_params['model2_models']:
                    for temp2 in experiment_params['temp2_temperatures']:
                        try:
                            result = run_grid(edge_pairs,
                                                    truth_edge_pairs,
                                                    oll_client,
                                                    experiment_params,
                                                    model, 
                                                    model2, 
                                                    temp, 
                                                    temp2,
                                                    write_lock,
                                                    pbar,
                                                    existing_data
                                                   )         
                            
                        except Exception as e:
                            print(f'ERROR: {e}:\n')
                            traceback.format_exc(e)
                        
    
                        pbar.update(1)
                        if result is not None:
                            experiments.append(result)
    
else:
    
    with ThreadPoolExecutor(max_workers=MAX_THREADS) as executor:
        with tqdm(total=total_combos, desc='Grid Combos') as pbar:
            for model in experiment_params['model1_models']:
                for temp in experiment_params['temp1_temperatures']:
                    for model2 in experiment_params['model2_models']:
                        for temp2 in experiment_params['temp2_temperatures']:
                            try:
                                futures.append( {
                                    (model, model2, temp, temp2, experiment_params['iterations']): executor.submit(run_grid, 
                                                                                                                    edge_pairs,
                                                                                                                    truth_edge_pairs,
                                                                                                                    oll_client,
                                                                                                                    experiment_params,
                                                                                                                    model, 
                                                                                                                    model2, 
                                                                                                                    temp, 
                                                                                                                    temp2,
                                                                                                                    write_lock,
                                                                                                                    pbar,
                                                                                                                    existing_data
                                                                                                                   )         
                                })
                            except Exception as e:
                                print(f'ERROR: {e}:\n')
                                traceback.format_exc(e)
                            
        #print(type(futures[0]))
        for future in as_completed(futures):
            result = future.result()
            pbar.update(1)
            if result is not None:
                experiments.append(result)
            

Total Combinations: 144, Total Iterations: 720


Grid Combos:   0%|          | 0/144 [00:00<?, ?it/s]

Iterations:   0%|          | 0/5 [00:00<?, ?it/s]

Prompt 1:   0%|          | 0/107 [00:00<?, ?it/s]

Prompt 2:   0%|          | 0/107 [00:00<?, ?it/s]

...

Prompt 1:   0%|          | 0/107 [00:00<?, ?it/s]

Prompt 2: 0it [00:00, ?it/s]

Prompt 3: 0it [00:00, ?it/s]

Iterations:   0%|          | 0/5 [00:00<?, ?it/s]

Prompt 1:   0%|          | 0/107 [00:00<?, ?it/s]

## Suggest Edges

In [57]:
#mlflow.openai.autolog()
mlflow.set_tracking_uri('http://localhost:5000')

edge_pairs = pairings # get_graph_edges(db, aql, graph_name, include_node_docs=True)
#edge_pairs = [edge for edge in edge_pairs if edge['_id'] == 'REFERENCES/20005']

experiment_params = {
    'partial_correct_threshold': 1, # The number of fields that can be wrong to be considered partially correct
    'quiet': True,
    'json_output_file': '../grid_results/suggested_results_111725_01.json'
}
model1 = GPT_OSS_120B_MODEL
model2 = LLAMA3_3_70B_MODEL
temp1 = 0.9
temp2 = 0.3

suggested_edge_responses = []
futures = []

MAX_THREADS = 2
BATCHES = 4
MAX_THREADS = min(MAX_THREADS, BATCHES)
write_lock = threading.Lock()

edge_pair_batches = edge_pairs if BATCHES <= 1 else [edge_pairs[i::BATCHES] for i in range(BATCHES)]

if MAX_THREADS == 1:
    for edge_batch in tqdm(edge_pair_batches, leave=False, desc='Batches'):
        try:
            
            result = suggest_edges(edge_batch,
                                    oll_client,
                                    experiment_params,
                                    model1, 
                                    model2, 
                                    temp1, 
                                    temp2,
                                    write_lock,
                                    src_node_key='src_node',
                                    pair_node_key='pair_node'
                                   )         
            
        except Exception as e:
            print(f'ERROR: {e}:\n')
            traceback.format_exc(e)
        
    
        if result is not None:
            suggested_edge_responses.append(result)
    
else:
    
    with ThreadPoolExecutor(max_workers=MAX_THREADS) as executor:
        with tqdm(edge_pair_batches, leave=False, desc='Batches') as pbar:
            for edge_batch in edge_pair_batches:
                try:
                    futures.append(  executor.submit(suggest_edges, 
                                                                        edge_batch,
                                                                        oll_client,
                                                                        experiment_params,
                                                                        model1, 
                                                                        model2, 
                                                                        temp1, 
                                                                        temp2,
                                                                        write_lock,
                                                                        src_node_key='src_node',
                                                                        pair_node_key='pair_node'
                                                                       )         
                    )
                except Exception as e:
                    print(f'ERROR: {e}:\n')
                    traceback.format_exc(e)
                                
            #print(type(futures[0]))
            for future in as_completed(futures):
                result = future.result()
                pbar.update(1)
                if result is not None:
                    suggested_edge_responses.append(result)
            

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Prompt 1:   0%|          | 0/200 [00:00<?, ?it/s]

Prompt 1:   0%|          | 0/200 [00:00<?, ?it/s]

ERROR: [Errno 104] Connection reset by peer
Skipping prompt...
ERROR: [Errno 104] Connection reset by peer
Skipping prompt...
🏃 View run gpt-oss:120b-0.9_llama3.3:70b-0.3__2025-11-18 16:30:26.167249 at: http://localhost:5000/#/experiments/438591178686409090/runs/6a95989c514342b6afc4ebbbaed8091d
🧪 View experiment at: http://localhost:5000/#/experiments/438591178686409090
🏃 View run gpt-oss:120b-0.9_llama3.3:70b-0.3__2025-11-18 16:30:26.106762 at: http://localhost:5000/#/experiments/438591178686409090/runs/bc8c0c5991ff40f983eca3faefe3ffa6
🧪 View experiment at: http://localhost:5000/#/experiments/438591178686409090


Prompt 1:   0%|          | 0/200 [00:00<?, ?it/s]

Prompt 1:   0%|          | 0/200 [00:00<?, ?it/s]

ERROR: [Errno 104] Connection reset by peer
Skipping prompt...
🏃 View run gpt-oss:120b-0.9_llama3.3:70b-0.3__2025-11-18 16:32:01.438753 at: http://localhost:5000/#/experiments/438591178686409090/runs/1ff4e339046541038a8643cdf1e985f2
🧪 View experiment at: http://localhost:5000/#/experiments/438591178686409090
ERROR: [Errno 104] Connection reset by peer
Skipping prompt...
🏃 View run gpt-oss:120b-0.9_llama3.3:70b-0.3__2025-11-18 16:32:01.452214 at: http://localhost:5000/#/experiments/438591178686409090/runs/cb03d0a4dfc84ba39e909f9499a710b7
🧪 View experiment at: http://localhost:5000/#/experiments/438591178686409090


TypeError: '>=' not supported between instances of 'ReadError' and 'int'

In [63]:
pd.options.display.max_columns = None
suggested_edge_responses_df = pd.read_json('../grid_results/suggested_results_111725_01.json', lines=True)
suggested_edge_responses = suggested_edge_responses_df.to_dict('records')
suggested_edges = [p3 for response in suggested_edge_responses for p3 in response['part3_responses'] if p3['explanation'] != 'NO CONNECTION' and p3['conn_strength'] >= conn_str_thresh]


In [66]:
#print(suggested_edge_responses[0])
suggested_edges = [p3 for response in suggested_edge_responses for p3 in response['part3_responses'] if p3['explanation'] != 'NO CONNECTION' and p3['conn_strength'] >= conn_str_thresh]
print(f'Suggesting {len(suggested_edges)} total edges:\n\n')
verified, denied = verify_edges(suggested_edges, db, auto_accept_partial=True)
print(f'Verified {len(verified)} edges; Denied {len(denied)} edges.')

Suggesting 239 total edges:




  0%|          | 0/239 [00:00<?, ?it/s]

{
    "src_node": {
        "_key": "obap_apt_001",
        "_id": "APTProfile/obap_apt_001",
        "description": "",
        "hours_spent": "",
        "artifact_location": "",
        "name": "obap_apt_001",
        "scenario_id": "OBAP",
        "apt_name": "APT29",
        "apt_number": "APT29",
        "mitre_tid": "G0016",
        "malware_samples": [
            "WellMess",
            "WellMail"
        ],
        "created_date": "2024-06-01"
    },
    "pair_node": {
        "_key": "T1574.001",
        "_id": "TTPArtifact/T1574.001",
        "tid": "T1574.001",
        "name": "Hijack Execution Flow: DLL",
        "description": "Adversaries may abuse dynamic-link library files (DLLs) in order to achieve persistence, escalate privileges, and evade defenses. DLLs are libraries that contain code and data that can be simultaneously utilized by multiple programs. While DLLs are not malicious by nature, they can be abused through mechanisms such as side-loading, hijacking searc

Verify this suggested edge? (y/n)   





{
    "src_node": {
        "_key": "obap_apt_001",
        "_id": "APTProfile/obap_apt_001",
        "description": "",
        "hours_spent": "",
        "artifact_location": "",
        "name": "obap_apt_001",
        "scenario_id": "OBAP",
        "apt_name": "APT29",
        "apt_number": "APT29",
        "mitre_tid": "G0016",
        "malware_samples": [
            "WellMess",
            "WellMail"
        ],
        "created_date": "2024-06-01"
    },
    "pair_node": {
        "_key": "T1584.001",
        "_id": "TTPArtifact/T1584.001",
        "tid": "T1584.001",
        "name": "Compromise Infrastructure: Domains",
        "description": "Adversaries may hijack domains and/or subdomains that can be used during targeting. Domain registration hijacking is the act of changing the registration of a domain name without the permission of the original registrant.(Citation: ICANNDomainNameHijacking) Adversaries may gain access to an email account for the person listed as the own

...




{
    "src_node": {
        "_key": "obap_deployment_001",
        "_id": "VMDeploymentArtifact/obap_deployment_001",
        "description": "",
        "hours_spent": "",
        "artifact_location": "",
        "name": "obap_deployment_001",
        "scenario_id": "OBAP",
        "platform": "VMware_vSphere",
        "access": "RDP/SSH",
        "deployed_vms": [
            {
                "vm_name": "DC01-WIN2019",
                "ip": "172.48.1.10",
                "os": "Windows Server 2019"
            },
            {
                "vm_name": "WORKSTATION-WIN10",
                "ip": "172.48.3.5",
                "os": "Windows 10"
            }
        ]
    },
    "pair_node": {
        "_key": "T1021.001",
        "_id": "TTPArtifact/T1021.001",
        "tid": "T1021.001",
        "name": "Remote Services: Remote Desktop Protocol",
        "description": "Adversaries may use [Valid Accounts](https://attack.mitre.org/techniques/T1078) to log into a computer using the

Verify this suggested edge? (y/n)   y





Verified 114 edges; Denied 125 edges.


  0%|          | 0/114 [00:00<?, ?it/s]

DocumentInsertError: [HTTP 404][ERR 1203] collection or view not found: COLLABORATION_WITH

In [74]:
add_verified_edges(verified, db)

  0%|          | 0/114 [00:00<?, ?it/s]

In [75]:
print(len(verified))
with open('../grid_results/new_edge_keys.txt', 'a') as file:
    unique_keys = set([edge['_key'] for edge in verified])
    print(len(unique_keys))
    fstr = '\n'.join(unique_keys)
    file.write(fstr)

114
64
